# Image Acquisition Script

This notebook handles the acquisition of human face images and converts them to 256x256 format.
The images will be saved in the `assets/raw_humanface/` folder for use in the data pipeline.


In [1]:
# Import required libraries
from PIL import Image
import os
import requests
from io import BytesIO
import time

In [2]:
# Configuration
NUM_IMAGES = 1000  # Number of images to download
OUTPUT_DIR = "../assets/raw_humanface"  # Output directory relative to scripts folder
IMAGE_SIZE = (256, 256)  # Target image size
SITE_URL = "https://thispersondoesnotexist.com" # Use this website to avoid privacy issues

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory created: {OUTPUT_DIR}")

Output directory created: ../assets/raw_humanface


In [3]:
def download_human_faces(num_images=NUM_IMAGES, output_dir=OUTPUT_DIR, image_size=IMAGE_SIZE):
    """
    Download human face images from image website and resize to specified dimensions.
    Skips download if the target number of images already exists.
    
    Args:
        num_images (int): Number of images to download
        output_dir (str): Directory to save images
        image_size (tuple): Target image size (width, height)
    """
    import glob
    
    # Check if images already exist
    existing_images = glob.glob(f"{output_dir}/*.jpg")
    num_existing = len(existing_images)
    
    if num_existing >= num_images:
        print(f"Found {num_existing} images already in {output_dir}")
        print(f"Target of {num_images} images already met. Skipping download.")
        return
    
    print(f"Found {num_existing} existing images")
    print(f"Starting download of {num_images - num_existing} additional human face images...")
    
    for i in range(num_existing, num_images):
        try:
            # Request image from thispersondoesnotexist.com
            resp = requests.get(
                SITE_URL, 
                headers={"User-Agent": "Mozilla/5.0"}
            )
            
            # Open image and convert to RGB
            img = Image.open(BytesIO(resp.content)).convert("RGB")
            
            # Resize to target dimensions
            img = img.resize(image_size, Image.Resampling.LANCZOS)
            
            # Save image with zero-padded 4-digit filename for better sorting
            img.save(f"{output_dir}/{i:04d}.jpg", "JPEG", quality=95)
            
            # Progress update
            if (i + 1) % 100 == 0:
                print(f"Downloaded {i + 1}/{num_images} images")
                
            # Small delay to avoid overwhelming the server
            time.sleep(0.1)
            
        except Exception as e:
            print(f"Error downloading image {i}: {e}")
            continue
    
    print(f"Successfully downloaded {num_images} human face images to {output_dir}")

In [4]:
# Execute the image download
if __name__ == "__main__":
    download_human_faces()

Found 0 existing images
Starting download of 1000 additional human face images...
Downloaded 100/1000 images
Downloaded 200/1000 images
Downloaded 300/1000 images
Downloaded 400/1000 images
Downloaded 500/1000 images
Downloaded 600/1000 images
Downloaded 700/1000 images
Downloaded 800/1000 images
Downloaded 900/1000 images
Downloaded 1000/1000 images
Successfully downloaded 1000 human face images to ../assets/raw_humanface


In [5]:
# Verify download results
import glob

def verify_downloads(output_dir=OUTPUT_DIR, expected_count=NUM_IMAGES):
    """Verify the downloaded images and display statistics."""
    
    # Count downloaded images
    image_files = glob.glob(f"{output_dir}/*.jpg")
    num_downloaded = len(image_files)
    
    print(f"Verification Results:")
    print(f"   Total images found: {num_downloaded}")
    print(f"   Expected images: {expected_count}")
    
    if num_downloaded >= expected_count:
        print(f"   Target achieved!")
    elif num_downloaded > 0:
        print(f"   Need {expected_count - num_downloaded} more images")
    else:
        print("   No images found!")
    
    if num_downloaded > 0:
        # Check a sample image for dimensions
        sample_img = Image.open(image_files[0])
        print(f"   Sample image dimensions: {sample_img.size}")
        print(f"   Sample image format: {sample_img.format}")
        print(f"   Images saved in: {os.path.abspath(output_dir)}")
        
        # Display first few filenames (sorted)
        sorted_files = sorted([os.path.basename(f) for f in image_files])
        print(f"   Sample filenames: {sorted_files[:5]}")

# Run verification
verify_downloads()

Verification Results:
   Total images found: 1000
   Expected images: 1000
   Target achieved!
   Sample image dimensions: (256, 256)
   Sample image format: JPEG
   Images saved in: /Users/ariesslin/northeastern_projects/ie7374/project/PerToon/assets/raw_humanface
   Sample filenames: ['0000.jpg', '0001.jpg', '0002.jpg', '0003.jpg', '0004.jpg']
